In [1]:
import sqlite3

# Connect to database
conn = sqlite3.connect("enrollment_tracker.db")
cursor = conn.cursor()
print("Connected to enrollment_tracker.db")

# Create students table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL UNIQUE
)
""")

# Create courses table
cursor.execute("""
CREATE TABLE IF NOT EXISTS courses (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_name TEXT NOT NULL UNIQUE
)
""")

# Create enrollments junction table
cursor.execute("""
CREATE TABLE IF NOT EXISTS enrollments (
    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id),
    UNIQUE (student_id, course_id)
)
""")

conn.commit()
print("Tables created successfully.")

Connected to enrollment_tracker.db
Tables created successfully.


In [2]:
# At least 5 predefined courses
course_list = [
    ("BS ECE",),
    ("BS EE",),
    ("BS CE",),
    ("BS COE",),
    ("BS ME",)
]

cursor.executemany("""
INSERT OR IGNORE INTO courses (course_name)
VALUES (?)
""", course_list)
conn.commit()

# Display available courses
cursor.execute("SELECT * FROM courses")
print("Available Courses:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

Available Courses:
  [1] BS ECE
  [2] BS EE
  [3] BS CE
  [4] BS COE
  [5] BS ME


In [3]:
student_list = [
    ("Anna Reyes",),
    ("Carlos Dela Cruz",),
    ("Mika Santos",),
    ("Juan Garcia",),
    ("Maria Lopez",),
    ("Pedro Reyes",),
    ("Sofia Cruz",),
    ("Luis Torres",),
    ("Elena Ramos",),
    ("Marco Villanueva",)
]

cursor.executemany("""
INSERT OR IGNORE INTO students (full_name)
VALUES (?)
""", student_list)
conn.commit()

# Display all students
cursor.execute("SELECT * FROM students")
print("Registered Students:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

Registered Students:
  [1] Anna Reyes
  [2] Carlos Dela Cruz
  [3] Mika Santos
  [4] Juan Garcia
  [5] Maria Lopez
  [6] Pedro Reyes
  [7] Sofia Cruz
  [8] Luis Torres
  [9] Elena Ramos
  [10] Marco Villanueva


In [4]:
# (student_id, course_id) — multiple students in multiple courses
enrollments = [
    (1, 1),  # Anna        → BS ECE
    (1, 3),  # Anna        → BS CE
    (2, 2),  # Carlos      → BS EE
    (2, 4),  # Carlos      → BS COE
    (3, 3),  # Mika        → BS CE
    (3, 5),  # Mika        → BS ME
    (4, 1),  # Juan        → BS ECE
    (4, 2),  # Juan        → BS EE
    (5, 1),  # Maria       → BS ECE
    (5, 4),  # Maria       → BS COE
    (6, 2),  # Pedro       → BS EE
    (6, 3),  # Pedro       → BS CE
    (7, 5),  # Sofia       → BS ME
    (8, 1),  # Luis        → BS ECE
    (9, 4),  # Elena       → BS COE
    (10, 5), # Marco       → BS ME
]

for enrollment in enrollments:
    try:
        cursor.execute("""
        INSERT INTO enrollments (student_id, course_id)
        VALUES (?, ?)
        """, enrollment)
        conn.commit()
    except sqlite3.IntegrityError:
        print(f"⚠️ Duplicate entry skipped: Student {enrollment[0]} → Course {enrollment[1]}")

print("Enrollments inserted successfully.")

Enrollments inserted successfully.


In [5]:
# Show available students
cursor.execute("SELECT * FROM students")
print("Students:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

student_id = int(input("\nEnter Student ID to view their courses: "))

cursor.execute("""
SELECT s.full_name, c.course_name
FROM enrollments e
JOIN students s ON e.student_id = s.student_id
JOIN courses c ON e.course_id = c.course_id
WHERE e.student_id = ?
""", (student_id,))

results = cursor.fetchall()
if results:
    print(f"\nCourses enrolled by {results[0][0]}:")
    for row in results:
        print(f"  - {row[1]}")
else:
    print("No enrollments found for this student.")

Students:
  [1] Anna Reyes
  [2] Carlos Dela Cruz
  [3] Mika Santos
  [4] Juan Garcia
  [5] Maria Lopez
  [6] Pedro Reyes
  [7] Sofia Cruz
  [8] Luis Torres
  [9] Elena Ramos
  [10] Marco Villanueva

Courses enrolled by Marco Villanueva:
  - BS ME


In [6]:
# Show available courses
cursor.execute("SELECT * FROM courses")
print("Courses:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

course_id = int(input("\nEnter Course ID to view enrolled students: "))

cursor.execute("""
SELECT c.course_name, s.full_name
FROM enrollments e
JOIN students s ON e.student_id = s.student_id
JOIN courses c ON e.course_id = c.course_id
WHERE e.course_id = ?
ORDER BY s.full_name ASC
""", (course_id,))

results = cursor.fetchall()
if results:
    print(f"\nStudents enrolled in {results[0][0]}:")
    for row in results:
        print(f"  - {row[1]}")
else:
    print("No students found for this course.")

Courses:
  [1] BS ECE
  [2] BS EE
  [3] BS CE
  [4] BS COE
  [5] BS ME

Students enrolled in BS ECE:
  - Anna Reyes
  - Juan Garcia
  - Luis Torres
  - Maria Lopez


In [7]:
# Show students and courses
cursor.execute("SELECT * FROM students")
print("Students:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

cursor.execute("SELECT * FROM courses")
print("\nCourses:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}")

student_id = int(input("\nEnter Student ID to enroll: "))
course_id  = int(input("Enter Course ID to enroll into: "))

try:
    cursor.execute("""
    INSERT INTO enrollments (student_id, course_id)
    VALUES (?, ?)
    """, (student_id, course_id))
    conn.commit()
    print("✅ Enrollment successful!")
except sqlite3.IntegrityError:
    print("⚠️ Warning: This student is already enrolled in that course!")

Students:
  [1] Anna Reyes
  [2] Carlos Dela Cruz
  [3] Mika Santos
  [4] Juan Garcia
  [5] Maria Lopez
  [6] Pedro Reyes
  [7] Sofia Cruz
  [8] Luis Torres
  [9] Elena Ramos
  [10] Marco Villanueva

Courses:
  [1] BS ECE
  [2] BS EE
  [3] BS CE
  [4] BS COE
  [5] BS ME
⚠️ Warning: This student is already enrolled in that course!


In [8]:
cursor.close()
conn.close()
print("Database connection closed.")

Database connection closed.
